# Demo of Optuna
This is a portion of the notebook for
### Chapter 10 – Building Neural Networks with PyTorch
**Hands-on Machine Learning with Scikit-Learn and Pytorch by Aurelien Geron, O'Reilly 2025**

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/ageron/handson-mlp/blob/main/10_neural_nets_with_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/ageron/handson-mlp/blob/main/10_neural_nets_with_pytorch.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

In [1]:
%pip install mlflow 

Note: you may need to restart the kernel to use updated packages.


# Setup

This project requires Python 3.10 or above:

In [2]:
import sys
import json

assert sys.version_info >= (3, 10)

It also requires Scikit-Learn ≥ 1.6.1:

In [3]:
from packaging.version import Version
import sklearn
import mlflow

assert Version(sklearn.__version__) >= Version("1.6.1")

/Users/ayoolanjoku/opt/anaconda3/envs/dlasgn1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
#  Init MLflow tracking
mlflow.set_tracking_uri(uri="http://127.0.0.1:5002")

# --- Create / set the experiment so all Optuna trials live together ---
EXPERIMENT_NAME = "104-hw03-optuna-pytorch"
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"MLflow tracking URI : {mlflow.get_tracking_uri()}")
print(f"Active experiment   : {EXPERIMENT_NAME}")

2026/02/21 00:26:12 INFO mlflow.tracking.fluent: Experiment with name '104-hw03-optuna-pytorch' does not exist. Creating a new experiment.


MLflow tracking URI : http://127.0.0.1:5002
Active experiment   : 104-hw03-optuna-pytorch


Are we using Colab or Kaggle?

In [5]:
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "kaggle_secrets" in sys.modules

If using Colab, a couple libraries are not pre-installed so we must install them manually:

In [6]:
if IS_COLAB:
    %pip install -q optuna torchmetrics

In [7]:
!pip install -q optuna torchmetrics

And of course we need PyTorch, specifically PyTorch ≥ 2.6.0:

In [8]:
import torch

assert Version(torch.__version__) >= Version("2.6.0")

### Harware Accelarot

In [9]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device


'mps'

Let us define the default font sizes to make the figures prettier:

In [10]:
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

In [11]:
import torchmetrics
import time

In [12]:
import torch.nn as nn

torch.manual_seed(42)  # to get reproducible results

# Building an Image Classifier with PyTorch

## Using TorchVision to Load the Dataset

In [13]:
import torchvision
import torchvision.transforms.v2 as T

toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

train_and_valid_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=True, download=True, transform=toTensor)
test_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=False, download=True, transform=toTensor)

torch.manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(
    train_and_valid_data, [55_000, 5_000])

In [14]:
from torch.utils.data import TensorDataset, DataLoader

In [15]:
torch.manual_seed(42)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

Each entry is a tuple (image, target):

In [16]:
X_sample, y_sample = train_data[0]

Each image has a shape \[channels, rows, columns\]. Grayscale images like in Fashion MNIST have a single channel (while RGB images have 3, and other types of images, such as satellite images, may have many more). Fashion images are grayscale and 28x28 pixels:

In [17]:
X_sample.shape

torch.Size([1, 28, 28])

In [18]:
X_sample.dtype

torch.float32

In [19]:
train_and_valid_data.classes[y_sample]

'Ankle boot'

## Building the Classifier

In [20]:
import torch.nn as nn

torch.manual_seed(42)  # to get reproducible results

In [21]:
class ImageClassifier(nn.Module):
    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs, n_hidden1),
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2),
            nn.ReLU(),
            nn.Linear(n_hidden2, n_classes)
        )

    def forward(self, X):
        return self.mlp(X)

torch.manual_seed(42)
model = ImageClassifier(n_inputs=1 * 28 * 28, n_hidden1=300, n_hidden2=100,
                        n_classes=10).to(device)
xentropy = nn.CrossEntropyLoss()

In [22]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()  # reset the metric at the beginning
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)  # update it at each iteration
    return metric.compute()  # compute the final result at the end

In [23]:
def train2(model, optimizer, criterion, metric, train_loader, valid_loader,
               n_epochs):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": [], "training_time": []}
    for epoch in range(n_epochs):
        start_time = time.time()
        total_loss = 0.
        metric.reset()
        for X_batch, y_batch in train_loader:
            model.train()
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        train_time = time.time() - start_time # calculate training time for the epoch
        history["training_time"].append(train_time)
        mean_loss = total_loss / len(train_loader) # average loss per batch
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}, "
              f"training time: {history['training_time'][-1]:.2f}s")
    return history

In [24]:
learning_rate = 0.4
n_epochs = 20

In [25]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
_ = train2(model, optimizer, xentropy, accuracy, train_loader, valid_loader,
           n_epochs)

Epoch 1/20, train loss: 0.6061, train metric: 0.7814, valid metric: 0.8402, training time: 8.96s
Epoch 2/20, train loss: 0.4062, train metric: 0.8493, valid metric: 0.8452, training time: 4.97s
Epoch 3/20, train loss: 0.3627, train metric: 0.8665, valid metric: 0.8522, training time: 4.89s
Epoch 4/20, train loss: 0.3351, train metric: 0.8763, valid metric: 0.8626, training time: 5.32s
Epoch 5/20, train loss: 0.3140, train metric: 0.8836, valid metric: 0.8758, training time: 5.65s
Epoch 6/20, train loss: 0.2985, train metric: 0.8886, valid metric: 0.8644, training time: 5.75s
Epoch 7/20, train loss: 0.2846, train metric: 0.8929, valid metric: 0.8758, training time: 5.20s
Epoch 8/20, train loss: 0.2730, train metric: 0.8975, valid metric: 0.8730, training time: 5.37s
Epoch 9/20, train loss: 0.2630, train metric: 0.9021, valid metric: 0.8808, training time: 4.89s
Epoch 10/20, train loss: 0.2519, train metric: 0.9043, valid metric: 0.8816, training time: 5.08s
Epoch 11/20, train loss: 0.24

In [26]:
model.eval()
X_new, y_new = next(iter(valid_loader))
X_new = X_new[:3].to(device)
with torch.no_grad():
    y_pred_logits = model(X_new)
y_pred = y_pred_logits.argmax(dim=1)  # index of the largest logit
y_pred

tensor([7, 4, 2], device='mps:0')

In [27]:
[train_and_valid_data.classes[index] for index in y_pred]

['Sneaker', 'Coat', 'Pullover']

Let's check whether the model made the correct predictions:

In [28]:
y_new[:3]

tensor([7, 4, 2])

All correct! 😃

In [29]:
import torch.nn.functional as F
y_proba = F.softmax(y_pred_logits, dim=1)
if device == "mps":
    y_proba = y_proba.cpu()
y_proba.round(decimals=3)

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.8540, 0.0000,
         0.1450],
        [0.0000, 0.0000, 0.0050, 0.0000, 0.9940, 0.0000, 0.0010, 0.0000, 0.0000,
         0.0000],
        [0.0020, 0.0000, 0.6470, 0.0000, 0.1150, 0.0000, 0.2360, 0.0000, 0.0000,
         0.0000]])

In [30]:
y_top4_values, y_top4_indices = torch.topk(y_pred_logits, k=4, dim=1)
y_top4_probas = F.softmax(y_top4_values, dim=1)
if device == "mps":
    y_top4_probas = y_top4_probas.cpu()
y_top4_probas.round(decimals=3)

tensor([[0.8540, 0.1450, 0.0000, 0.0000],
        [0.9940, 0.0050, 0.0010, 0.0000],
        [0.6470, 0.2360, 0.1150, 0.0020]])

In [31]:
y_top4_indices

tensor([[7, 9, 5, 8],
        [4, 2, 6, 0],
        [2, 6, 4, 0]], device='mps:0')

In [32]:
sum([param.numel() for param in model.parameters()])

266610

# Hyperparameter Tuning using Optuna

In [33]:
import time
import optuna

def objective(trial):
    with mlflow.start_run(run_name=f"trial-{trial.number}-train-objective1-{(time.asctime())}"):
        mlflow.log_param("device", device)
        mlflow.set_tags({
            "optuna_trial_number": trial.number,
            "framework": "pytorch",
            "task": "fashion_mnist_classification",
            "mlflow.user": "ayoolanjoku"
        })
        learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
        n_hidden = trial.suggest_int("n_hidden", 20, 300)
        print(trial.params)
        mlflow.log_params(trial.params)
        model = ImageClassifier(n_inputs=1 * 28 * 28, n_hidden1=n_hidden,
                                n_hidden2=n_hidden, n_classes=10).to(device)
        optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
        xentropy = nn.CrossEntropyLoss()
        accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10)
        accuracy = accuracy.to(device)
        start_time = time.time()
        history = train2(model, optimizer, xentropy, accuracy, train_loader,
                        valid_loader, n_epochs=10)
        end_time = time.time()
        mlflow.log_text(json.dumps(history, indent=2), "history.json")
        mlflow.pytorch.log_model(model, artifact_path="model")
        mlflow.log_param("EPOCH COUNT", len(history["train_losses"]))
        mlflow.log_metric("final_train_loss", history["train_losses"][-1])



        validation_accuracy = max(history["valid_metrics"])
        train_loss = min(history["train_losses"])
        average_train_loss = sum(history["train_losses"]) / len(history["train_losses"])
        
        mlflow.log_metric("validation_accuracy", validation_accuracy)
        mlflow.log_metric("final_train_loss", train_loss)
        mlflow.log_metric("average_train_loss", average_train_loss)
        mlflow.log_metric("training_time", end_time - start_time)
        return validation_accuracy

In [34]:
def main1():
    torch.manual_seed(42)
    sampler = optuna.samplers.TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=5)


    print(f"\nStudy Statistics:")
    print(f"  Total trials: {len(study.trials)}")
    
    completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
    failed = [t for t in study.trials if t.state == optuna.trial.TrialState.FAIL]
    
    print(f"  Completed: {len(completed)}")
    print(f"  Pruned: {len(pruned)}")
    print(f"  Failed: {len(failed)}")
    
    if completed:
        print(f"\nBest trial:")
        print(f"  Value: {study.best_value}")
        print(f"  Params: {study.best_params}")
    else:
        print("\n  No trials completed successfully!")
        if failed:
            print(f"  {len(failed)} trial(s) failed - check error messages above")
        if pruned:
            print(f"  {len(pruned)} trial(s) were pruned - consider relaxing pruner settings")
main1()



[I 2026-02-21 00:28:12,245] A new study created in memory with name: no-name-80a3c3a2-018a-4a48-85ad-f2afccb0d593


{'learning_rate': 0.00031489116479568613, 'n_hidden': 287}
Epoch 1/10, train loss: 2.2769, train metric: 0.1471, valid metric: 0.1860, training time: 5.77s
Epoch 2/10, train loss: 2.2093, train metric: 0.2794, valid metric: 0.3500, training time: 5.20s
Epoch 3/10, train loss: 2.1164, train metric: 0.4109, valid metric: 0.4554, training time: 4.91s
Epoch 4/10, train loss: 1.9776, train metric: 0.5137, valid metric: 0.5560, training time: 4.88s
Epoch 5/10, train loss: 1.7867, train metric: 0.5826, valid metric: 0.6026, training time: 4.92s
Epoch 6/10, train loss: 1.5775, train metric: 0.6184, valid metric: 0.6228, training time: 4.97s
Epoch 7/10, train loss: 1.3978, train metric: 0.6288, valid metric: 0.6326, training time: 4.98s
Epoch 8/10, train loss: 1.2605, train metric: 0.6360, valid metric: 0.6372, training time: 5.22s
Epoch 9/10, train loss: 1.1572, train metric: 0.6467, valid metric: 0.6426, training time: 4.81s


2026/02/21 00:29:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/21 00:29:06 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 10/10, train loss: 1.0782, train metric: 0.6537, valid metric: 0.6436, training time: 4.93s


[I 2026-02-21 00:29:08,660] Trial 0 finished with value: 0.6435999870300293 and parameters: {'learning_rate': 0.00031489116479568613, 'n_hidden': 287}. Best is trial 0 with value: 0.6435999870300293.


...

🏃 View run trial-3-train-objective1-Sat Feb 21 00:30:59 2026 at: http://127.0.0.1:5002/#/experiments/6/runs/41855a49f42d4a158bad33d4b0bd15a3
🧪 View experiment at: http://127.0.0.1:5002/#/experiments/6
{'learning_rate': 0.002537815508265664, 'n_hidden': 218}
Epoch 1/10, train loss: 1.8379, train metric: 0.4869, valid metric: 0.6208, training time: 4.93s
Epoch 2/10, train loss: 0.9751, train metric: 0.6666, valid metric: 0.6978, training time: 4.72s
Epoch 3/10, train loss: 0.7608, train metric: 0.7253, valid metric: 0.7416, training time: 4.78s
Epoch 4/10, train loss: 0.6704, train metric: 0.7638, valid metric: 0.7720, training time: 4.67s
Epoch 5/10, train loss: 0.6108, train metric: 0.7913, valid metric: 0.7906, training time: 4.78s
Epoch 6/10, train loss: 0.5687, train metric: 0.8054, valid metric: 0.8050, training time: 5.13s
Epoch 7/10, train loss: 0.5386, train metric: 0.8163, valid metric: 0.8082, training time: 4.80s
Epoch 8/10, train loss: 0.5158, train metric: 0.8243, valid met

2026/02/21 00:32:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/21 00:32:45 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 10/10, train loss: 0.4842, train metric: 0.8331, valid metric: 0.8092, training time: 4.82s


[I 2026-02-21 00:32:47,680] Trial 4 finished with value: 0.8220000267028809 and parameters: {'learning_rate': 0.002537815508265664, 'n_hidden': 218}. Best is trial 1 with value: 0.854200005531311.


🏃 View run trial-4-train-objective1-Sat Feb 21 00:31:53 2026 at: http://127.0.0.1:5002/#/experiments/6/runs/ac4c3a0e91e245f58ccce9f441268bac
🧪 View experiment at: http://127.0.0.1:5002/#/experiments/6

Study Statistics:
  Total trials: 5
  Completed: 5
  Pruned: 0
  Failed: 0

Best trial:
  Value: 0.854200005531311
  Params: {'learning_rate': 0.008471801418819975, 'n_hidden': 188}


In [35]:
import time

def objective(trial, train_loader, valid_loader):
    with mlflow.start_run(run_name=f"trial-{trial.number}-objective2-time-{(time.time())}"): # add time to the run name to make it unique and more informative
        mlflow.set_tags({
            "optuna_trial_number": trial.number,
            "framework": "pytorch",
            "task": "fashion_mnist_classification",
            "mlflow.user": "ayoolanjoku"
        })
        learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
        n_hidden = trial.suggest_int("n_hidden", 20, 300)
        model = ImageClassifier(n_inputs=1 * 28 * 28, n_hidden1=n_hidden,
                                n_hidden2=n_hidden, n_classes=10).to(device)
        optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
        xentropy = nn.CrossEntropyLoss()
        mlflow.log_params(trial.params)
        accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10)
        accuracy = accuracy.to(device)
        best_validation_accuracy = 0.0
        for epoch in range(n_epochs):
            history = train2(model, optimizer, xentropy, accuracy, train_loader,
                            valid_loader, n_epochs=1)
            train_accuracy = history["train_metrics"][-1]
            train_loss = history["train_losses"][-1]
            validation_accuracy = max(history["valid_metrics"])
            if validation_accuracy > best_validation_accuracy:
                best_validation_accuracy = validation_accuracy
            trial.report(validation_accuracy, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        mlflow.log_text(json.dumps(history, indent=2), "history.json")
        mlflow.log_param("EPOCH COUNT", epoch + 1)
        mlflow.log_metric("final_train_loss", train_loss)
        mlflow.log_metric("average_train_loss", sum(history["train_losses"]) / len(history["train_losses"]))
        mlflow.log_metric("average_train_time", sum(history["training_time"]) / len(history["training_time"]))
        mlflow.log_metric("final_train_accuracy", train_accuracy)
        mlflow.log_metric("validation_accuracy", best_validation_accuracy) 
        # mlflow.log_metric("training_time", end_time - start_time)

        mlflow.pytorch.log_model(model, artifact_path="model")
    return best_validation_accuracy

In [36]:
objective_with_data = lambda trial: objective(
    trial, train_loader=train_loader, valid_loader=valid_loader)

In [37]:
from functools import partial

objective_with_data = partial(objective, train_loader=train_loader,
                              valid_loader=valid_loader)

In [38]:
# torch.manual_seed(42)
# sampler = optuna.samplers.TPESampler(seed=42)
# pruner = optuna.pruners.MedianPruner()
# study = optuna.create_study(direction="maximize", sampler=sampler,
#                             pruner=pruner)
# study.optimize(objective_with_data, n_trials=20)

In [39]:
# study.best_value

In [40]:
# study.best_params

In [41]:
def main():
    torch.manual_seed(42)
    sampler = optuna.samplers.TPESampler(seed=42)
    pruner = optuna.pruners.MedianPruner()
    study = optuna.create_study(direction="maximize", sampler=sampler,
                                pruner=pruner)
    study.optimize(objective_with_data, n_trials=20)
    
main()

[I 2026-02-21 00:32:47,726] A new study created in memory with name: no-name-1d0409b0-8e57-452d-8bf7-46c54c479a3a


Epoch 1/1, train loss: 2.2769, train metric: 0.1471, valid metric: 0.1860, training time: 4.96s
Epoch 1/1, train loss: 2.2093, train metric: 0.2794, valid metric: 0.3500, training time: 5.04s
Epoch 1/1, train loss: 2.1164, train metric: 0.4109, valid metric: 0.4554, training time: 4.99s
Epoch 1/1, train loss: 1.9776, train metric: 0.5137, valid metric: 0.5560, training time: 5.16s
Epoch 1/1, train loss: 1.7867, train metric: 0.5826, valid metric: 0.6026, training time: 4.98s
Epoch 1/1, train loss: 1.5775, train metric: 0.6184, valid metric: 0.6228, training time: 5.06s
Epoch 1/1, train loss: 1.3978, train metric: 0.6288, valid metric: 0.6326, training time: 5.18s
Epoch 1/1, train loss: 1.2605, train metric: 0.6360, valid metric: 0.6372, training time: 5.03s
Epoch 1/1, train loss: 1.1572, train metric: 0.6467, valid metric: 0.6426, training time: 4.87s
Epoch 1/1, train loss: 1.0782, train metric: 0.6537, valid metric: 0.6436, training time: 4.84s
Epoch 1/1, train loss: 1.0162, train met

2026/02/21 00:34:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/21 00:34:39 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 1/1, train loss: 0.7647, train metric: 0.7196, valid metric: 0.7082, training time: 5.05s


[I 2026-02-21 00:34:41,520] Trial 0 finished with value: 0.7089999914169312 and parameters: {'learning_rate': 0.00031489116479568613, 'n_hidden': 287}. Best is trial 0 with value: 0.7089999914169312.


...

Epoch 1/1, train loss: 1.8006, train metric: 0.4596, valid metric: 0.6054, training time: 6.41s
🏃 View run trial-18-objective2-time-1771653583.045554 at: http://127.0.0.1:5002/#/experiments/6/runs/cf1b65ec2de64a5f92995007006cfeaf
🧪 View experiment at: http://127.0.0.1:5002/#/experiments/6
Epoch 1/1, train loss: 0.7267, train metric: 0.7427, valid metric: 0.8248, training time: 6.30s
Epoch 1/1, train loss: 0.4580, train metric: 0.8355, valid metric: 0.8458, training time: 6.18s
Epoch 1/1, train loss: 0.4053, train metric: 0.8533, valid metric: 0.8570, training time: 6.20s
Epoch 1/1, train loss: 0.3747, train metric: 0.8645, valid metric: 0.8496, training time: 6.16s
Epoch 1/1, train loss: 0.3530, train metric: 0.8712, valid metric: 0.8618, training time: 6.15s
Epoch 1/1, train loss: 0.3357, train metric: 0.8779, valid metric: 0.8754, training time: 6.19s
Epoch 1/1, train loss: 0.3193, train metric: 0.8826, valid metric: 0.8706, training time: 6.06s
Epoch 1/1, train loss: 0.3076, train m

2026/02/21 01:02:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/21 01:02:02 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 1/1, train loss: 0.2148, train metric: 0.9203, valid metric: 0.8886, training time: 6.29s


[I 2026-02-21 01:02:05,444] Trial 19 finished with value: 0.8938000202178955 and parameters: {'learning_rate': 0.035092247138920826, 'n_hidden': 235}. Best is trial 19 with value: 0.8938000202178955.


🏃 View run trial-19-objective2-time-1771653589.9875672 at: http://127.0.0.1:5002/#/experiments/6/runs/b0a4a1af488d4895acc3b13f8f4c6b81
🧪 View experiment at: http://127.0.0.1:5002/#/experiments/6
